In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import seaborn as sns
import os
import kagglehub
from pathlib import Path

print("Setting up movie recommender system...")
print("Downloading MovieLens dataset via KaggleHub...")

# Function to download and load MovieLens dataset
def load_movielens_data():
    """Download and load MovieLens dataset using KaggleHub"""
    try:
        # Try to download MovieLens small dataset first (preferred)
        dataset_path = kagglehub.dataset_download("shubhammehta21/movie-lens-small-latest-dataset")
        print(f"Downloaded dataset to: {dataset_path}")
        
        # List available files
        files = os.listdir(dataset_path)
        print(f"Available files: {files}")
        
        # Look for movies and ratings files
        movies_file = None
        ratings_file = None
        
        for file in files:
            if 'movies' in file.lower() and file.endswith('.csv'):
                movies_file = os.path.join(dataset_path, file)
            elif 'ratings_small' in file.lower() and file.endswith('.csv'):
                ratings_file = os.path.join(dataset_path, file)  # Prefer ratings_small
            elif 'ratings' in file.lower() and file.endswith('.csv') and ratings_file is None:
                ratings_file = os.path.join(dataset_path, file)  # Fallback to ratings
        
        if not movies_file or not ratings_file:
            raise FileNotFoundError("Required CSV files not found in dataset")
        
        print(f"Loading movies from: {os.path.basename(movies_file)}")
        print(f"Loading ratings from: {os.path.basename(ratings_file)}")
        
        movies = pd.read_csv(movies_file)
        ratings = pd.read_csv(ratings_file)
        
        return movies, ratings
        
    except Exception as e:
        print(f"KaggleHub download failed: {e}")
        print("Trying alternative dataset...")
        
        try:
            # Try alternative dataset
            dataset_path = kagglehub.dataset_download("grouplens/movielens-20m-dataset")
            print(f"Downloaded alternative dataset to: {dataset_path}")
            
            movies = pd.read_csv(os.path.join(dataset_path, "movies.csv"))
            ratings = pd.read_csv(os.path.join(dataset_path, "ratings.csv"))
            
            return movies, ratings
            
        except Exception as e2:
            print(f"Alternative dataset also failed: {e2}")
            print("\n=== MANUAL SETUP REQUIRED ===")
            print("Please download MovieLens dataset manually:")
            print("1. Visit: https://www.kaggle.com/datasets/shubhammehta21/movie-lens-small-latest-dataset")
            print("2. Download and extract movies.csv and ratings.csv")
            print("3. Place files in current directory")
            print("4. Restart notebook")
            
            # Check if files exist locally as fallback
            if os.path.exists('movies.csv') and os.path.exists('ratings.csv'):
                print("\nFound local CSV files, using them...")
                movies = pd.read_csv('movies.csv')
                ratings = pd.read_csv('ratings.csv')
                return movies, ratings
            elif os.path.exists('movies.csv') and os.path.exists('ratings_small.csv'):
                print("\nFound local CSV files (with ratings_small), using them...")
                movies = pd.read_csv('movies.csv')
                ratings = pd.read_csv('ratings_small.csv')
                return movies, ratings
            else:
                raise FileNotFoundError("No dataset available. Please download manually.")

# Load the data
movies, ratings = load_movielens_data()
print(f"\nDataset loaded successfully!")
print(f"Movies shape: {movies.shape}")
print(f"Ratings shape: {ratings.shape}")


In [2]:
movies['title'] = movies['title'].str.lower()


In [3]:
movies.head()


,movieId,title,genres
0,1,toy story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,jumanji (1995),Adventure|Children|Fantasy
2,3,grumpier old men (1995),Comedy|Romance
3,4,waiting to exhale (1995),Comedy|Drama|Romance
4,5,father of the bride part ii (1995),Comedy


In [4]:
ratings.head()


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [5]:
final_dataset = ratings.pivot(
    index='movieId', columns='userId', values='rating')
final_dataset.head()


In [ ]:
final_dataset.fillna(0, inplace=True)
final_dataset.head()


userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,0.0,0.0,4.0,0.0,4.5,0.0,0.0,0.0,...,4.0,0.0,4.0,3.0,4.0,2.5,4.0,2.5,3.0,5.0
2,0.0,0.0,0.0,0.0,0.0,4.0,0.0,4.0,0.0,0.0,...,0.0,4.0,0.0,5.0,3.5,0.0,0.0,2.0,0.0,0.0
3,4.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
no_user_voted = ratings.groupby('movieId')['rating'].agg('count')
no_movies_voted = ratings.groupby('userId')['rating'].agg('count')


In [ ]:
f, ax = plt.subplots(1, 1, figsize=(16, 4))
# ratings['rating'].plot(kind='hist')
plt.scatter(no_user_voted.index, no_user_voted, color='mediumseagreen')
plt.axhline(y=10, color='r')
plt.xlabel('MovieId')
plt.ylabel('No. of users voted')
plt.show()


In [ ]:
final_dataset = final_dataset.loc[:,
                                  no_movies_voted[no_movies_voted > 50].index]
final_dataset


userId,1,4,6,7,10,11,15,16,17,18,...,600,601,602,603,604,605,606,607,608,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,0.0,4.5,0.0,0.0,2.5,0.0,4.5,3.5,...,2.5,4.0,0.0,4.0,3.0,4.0,2.5,4.0,2.5,5.0
2,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,...,4.0,0.0,4.0,0.0,5.0,3.5,0.0,0.0,2.0,0.0
3,4.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0
4,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2.5,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193581,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
193583,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
193585,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
sample = np.array([[0, 0, 3, 0, 0], [4, 0, 0, 0, 2], [0, 0, 0, 0, 1]])
sparsity = 1.0 - (np.count_nonzero(sample) / float(sample.size))
print(sparsity)


0.7333333333333334


In [ ]:
csr_sample = csr_matrix(sample)
print(csr_sample)


  (0, 2)	3
  (1, 0)	4
  (1, 4)	2
  (2, 4)	1


In [ ]:
csr_data = csr_matrix(final_dataset.values)
final_dataset.reset_index(inplace=True)


In [ ]:
knn = NearestNeighbors(metric='cosine', algorithm='brute',
                       n_neighbors=20, n_jobs=-1)
knn.fit(csr_data)


NearestNeighbors(algorithm='brute', metric='cosine', n_jobs=-1, n_neighbors=20)

In [ ]:
def get_movie_recommendation(movie_name):
    n_movies_to_reccomend = 10
    movie_list = movies[movies['title'].str.contains(movie_name)]
    if len(movie_list):
        movie_idx = movie_list.iloc[0]['movieId']
        movie_idx = final_dataset[final_dataset['movieId']
                                  == movie_idx].index[0]
        distances, indices = knn.kneighbors(
            csr_data[movie_idx], n_neighbors=n_movies_to_reccomend+1)
        rec_movie_indices = sorted(list(zip(indices.squeeze().tolist(
        ), distances.squeeze().tolist())), key=lambda x: x[1])[:0:-1]
        recommend_frame = []
        for val in rec_movie_indices:
            movie_idx = final_dataset.iloc[val[0]]['movieId']
            idx = movies[movies['movieId'] == movie_idx].index
            recommend_frame.append(
                {'Title': movies.iloc[idx]['title'].values[0], 'Distance': val[1]})
        df = pd.DataFrame(recommend_frame, index=range(
            1, n_movies_to_reccomend+1))
        return df
    else:
        return "No movies found. Please check your input"


In [ ]:
get_movie_recommendation('aladdin')


'No movies found. Please check your input'

In [ ]:

message ='helloworld'